# Preparing the Forecasting Dataset

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
data_path = Path("../data/raw/RELIANCE.csv")
df = pd.read_csv(data_path)
df["Date"] = pd.to_datetime(df["Date"])
df.head()

,index,Date,Adj Close,Close,High,Low,Open,Volume
0,0,2015-01-01,189.125381,202.959000,203.896194,201.987518,202.593277,2963643
1,1,2015-01-02,188.624817,202.421829,204.821960,202.136108,203.004715,7331366
2,2,2015-01-05,186.558685,200.204575,203.644760,199.804550,202.296112,10103941
3,3,2015-01-06,178.091827,191.118393,199.553116,190.181198,198.867371,18627980
4,4,2015-01-07,181.968475,195.278610,196.307236,191.324127,191.346985,20720312


Create the prediction target

In [5]:
df["Target"] = df["Close"].shift(-1)
df[["Date", "Close", "Target"]].head()

,Date,Close,Target
0,2015-01-01,202.959000,202.421829
1,2015-01-02,202.421829,200.204575
2,2015-01-05,200.204575,191.118393
3,2015-01-06,191.118393,195.278610
4,2015-01-07,195.278610,192.478470


Create lag features

lag features tells about previous data

ex: lag5 = 5 previous days closing prices

In [6]:
df["lag1"] = df["Close"].shift(1)
df["lag2"] = df["Close"].shift(2)
df["lag3"] = df["Close"].shift(3)
df["lag5"] = df["Close"].shift(5)
df["lag10"] = df["Close"].shift(10)

df[["Date","Close","lag1","lag2","lag3","lag5","lag10","Target"]].head(12)

,Date,Close,lag1,lag2,lag3,lag5,lag10,Target
0,2015-01-01,202.959000,NaN,NaN,NaN,NaN,NaN,202.421829
1,2015-01-02,202.421829,202.959000,NaN,NaN,NaN,NaN,200.204575
2,2015-01-05,200.204575,202.421829,202.959000,NaN,NaN,NaN,191.118393
3,2015-01-06,191.118393,200.204575,202.421829,202.959000,NaN,NaN,195.278610
4,2015-01-07,195.278610,191.118393,200.204575,202.421829,NaN,NaN,192.478470
5,2015-01-08,192.478470,195.278610,191.118393,200.204575,202.959000,NaN,196.650101
6,2015-01-09,196.650101,192.478470,195.278610,191.118393,202.421829,NaN,194.364273
7,2015-01-12,194.364273,196.650101,192.478470,195.278610,200.204575,NaN,192.729904
8,2015-01-13,192.729904,194.364273,196.650101,192.478470,191.118393,NaN,190.855530
9,2015-01-14,190.855530,192.729904,194.364273,196.650101,195.278610,NaN,197.553009


create daily returns

In [7]:
df["Return"] = df["Close"].pct_change()
df[["Date", "Close", "Return"]].head()

,Date,Close,Return
0,2015-01-01,202.959000,NaN
1,2015-01-02,202.421829,-0.002647
2,2015-01-05,200.204575,-0.010954
3,2015-01-06,191.118393,-0.045384
4,2015-01-07,195.278610,0.021768


Calculate rolling features

In [9]:
df["MA5"] = df["Close"].rolling(window=5).mean()
df["MA20"] = df["Close"].rolling(window=20).mean()

df[["Date", "Close", "MA5", "MA20"]].tail()

,Date,Close,MA5,MA20
2906,2026-10-02,1167.699951,1180.399976,1235.050000
2907,2026-10-05,1186.400024,1178.159985,1228.895001
2908,2026-10-06,1218.000000,1185.359985,1225.050000
2909,2026-10-07,1207.699951,1189.499976,1221.484998
2910,2026-10-08,1178.000000,1191.559985,1216.684998


20-day volatility

In [10]:
df["Volatility20"] = df["Return"].rolling(window=20).std()
df[["Date", "Return", "Volatility20"]].tail()

,Date,Return,Volatility20
2906,2026-10-02,0.000000,0.010869
2907,2026-10-05,0.016014,0.011902
2908,2026-10-06,0.026635,0.013711
2909,2026-10-07,-0.008457,0.013601
2910,2026-10-08,-0.024592,0.014451


Removing the initial rows that do not have enough historical data to calculate all features.

In [11]:
df = df.dropna().reset_index(drop=True)

In [12]:
df.head()

,index,Date,Adj Close,Close,High,Low,Open,Volume,Target,lag1,lag2,lag3,lag5,lag10,Return,MA5,MA20,Volatility20
0,20,2015-01-30,194.951019,209.210754,212.868088,207.782104,212.582352,21746801,207.564957,212.445206,207.290649,203.221878,202.010391,197.553009,-0.015225,206.979782,200.206291,0.020023
1,21,2015-02-02,193.417374,207.564957,210.273666,207.027786,209.393616,19302710,214.319595,209.210754,212.445206,207.290649,202.730423,198.798798,-0.007867,207.946689,200.463448,0.020116
2,22,2015-02-03,199.711624,214.319595,215.062485,208.742157,208.742157,17092333,212.308060,207.564957,209.210754,212.445206,203.221878,201.073196,0.032542,210.166232,201.169199,0.021035
3,23,2015-02-04,197.837219,212.308060,215.736801,211.645172,214.662460,14072894,211.142288,214.319595,207.564957,209.210754,207.290649,206.422043,-0.009386,211.169714,202.228682,0.017932
4,24,2015-02-05,196.750900,211.142288,214.845322,209.747925,213.953857,12365643,207.987839,212.308060,214.319595,207.564957,212.445206,207.267792,-0.005491,210.909131,203.021866,0.017659


Select the forecasting features

In [14]:
features = ["Open","High","Low","Close","Volume","Return",
            "lag1","lag2","lag3","lag5","lag10",
            "MA5","MA20","Volatility20"]
target = "Target"

X = df[features]
y = df[target]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (2890, 14)
Target: (2890,)


Split the data into train and test data

In [15]:
splitidx = int(len(df) * 0.8)

Xtrain = X.iloc[:splitidx]
Xtest = X.iloc[splitidx:]
ytrain = y.iloc[:splitidx]
ytest = y.iloc[splitidx:]

print("Training data:", Xtrain.shape)
print("Testing data:", Xtest.shape)

Training data: (2312, 14)
Testing data: (578, 14)


In [17]:
print("Training period Interval:")
print(df.iloc[:splitidx]["Date"].min())
print(df.iloc[:splitidx]["Date"].max())

print("\nTesting period Interval:")
print(df.iloc[splitidx:]["Date"].min())
print(df.iloc[splitidx:]["Date"].max())

Training period Interval:
2015-01-30 00:00:00
2024-06-14 00:00:00

Testing period Interval:
2024-06-18 00:00:00
2026-10-07 00:00:00


Save the processed dataset

In [18]:
processed_path = Path("../data/processed/RELIANCE_processed.csv")
processed_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(processed_path, index=False)
print(f"Processed data saved to: {processed_path}")

Processed data saved to: ..\data\processed\RELIANCE_processed.csv
